# RAG with Unified Lineage — Working Demo (v0.0.4)

End-to-end: a single trace that spans data operations and LLM calls, with cross-domain links attributed automatically.

**What's new in v0.0.4:** no manual `register_object_identity` or `record_llm_input` calls. AutoLineage's new `register_assign_id_callback` API wires into the RudriQ linker; the OpenAI SDK is monkey-patched to capture inputs by span_id; and a small pandas lid-propagation patch carries lineage IDs through `df['col'].tolist()` so list inputs to LLM calls match upstream DataFrames automatically. The user just imports `rudriq.auto` and writes idiomatic pandas + openai code.

We use an httpx mock transport so the openai call runs offline. The instrumentation pattern is identical for real OpenAI calls.

## 1. Activate RudriQ

One import. This wires AutoLineage's `assign_id` callback into the linker (auto-registering object identity), monkey-patches the OpenAI SDK (auto-capturing LLM inputs by span_id), and patches `pd.DataFrame.__getitem__` and `pd.Series.tolist` (propagating lineage IDs through column-derived lists).

In [ ]:
import rudriq.auto

## 2. OTel setup

In production with Traceloop/OpenLLMetry installed, this is what `Traceloop.init()` would do for you. Without it, we set up a TracerProvider with the RudriQ SpanProcessor manually. This is the only RudriQ-specific code the notebook contains.

In [ ]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from rudriq.processors import RudriQSpanProcessor

provider = TracerProvider()
rudriq_proc = RudriQSpanProcessor()
provider.add_span_processor(rudriq_proc)
trace.set_tracer_provider(provider)
tracer = trace.get_tracer('rag-demo')
print(f'RudriQ run_id: {rudriq_proc.run_id}')

## 3. Run the pipeline — pandas + openai, no manual instrumentation

AutoLineage's pandas hooks fire on `read_csv` and the filter, and through the new callback API they automatically register the object identity of each output with the RudriQ linker. The lid-propagation patch then carries the filter's lineage ID through `df['text'].tolist()` so the resulting list — what we actually pass to OpenAI — is matchable. The OpenAI SDK call is monkey-patched to capture its `input=` parameter into the span side-channel.

In [ ]:
import io, pandas as pd

csv = io.StringIO(
    'id,lang,text\n'
    '1,en,doc about machine learning\n'
    '2,fr,document sur lapprentissage\n'
    '3,en,doc about retrieval\n'
    '4,en,doc about graphs\n'
)
docs = pd.read_csv(csv)
docs = docs[docs['lang'] == 'en']
print(f'Filtered to {len(docs)} English docs.')

In [ ]:
# Mock OpenAI backend so the call runs offline. In production this is a
# real OpenAI client — same code path; same auto-capture works.
import httpx, json
from openai import OpenAI

def fake_openai(request):
    n = len(json.loads(request.content)['input']) if request.content else 1
    return httpx.Response(200, json={
        'object': 'list', 'model': 'text-embedding-3-small',
        'data': [{'object': 'embedding', 'index': i, 'embedding': [0.1] * 8} for i in range(n)],
        'usage': {'prompt_tokens': 4 * n, 'total_tokens': 4 * n},
    })
client = OpenAI(api_key='sk-fake-offline-demo',
                http_client=httpx.Client(transport=httpx.MockTransport(fake_openai)))

# Production note: with Traceloop installed, this span is created
# automatically by OpenLLMetry's openai instrumentor. Without it,
# we wrap the call in a span ourselves so the SpanProcessor sees it.
with tracer.start_as_current_span('openai.embeddings.create') as span:
    span.set_attribute('gen_ai.system', 'openai')
    span.set_attribute('gen_ai.request.model', 'text-embedding-3-small')
    embeddings = client.embeddings.create(
        model='text-embedding-3-small',
        input=docs['text'].tolist(),
    )
print(f'Got {len(embeddings.data)} embeddings.')

## 4. Inspect the unified trace

The SpanProcessor persisted the LLM span. The auto-capture wrapper stashed the `tolist()` output against this span's id. The lid-propagation patch ensured that list inherited the filtered DataFrame's lineage ID. The linker matched them at `on_end` and emitted a cross-domain `LINEAGE_LINK` edge whose `parent_id` points to AutoLineage's tracker.

**v0.0.4 scope note.** RudriQ's DuckDB stores the LLM trace + cross-domain edges. Data-side nodes still live in AutoLineage's tracker (a separate store). v0.0.5 mirrors AutoLineage's records into RudriQ's storage so a single `load_run` returns the full graph. For now, we resolve the parent lid against AutoLineage to display the chain.

In [ ]:
from rudriq.storage import get_default_storage
import autolineage.auto as al_auto

graph = get_default_storage().load_run(rudriq_proc.run_id)
al_tracker = al_auto.get_tracker()

print(f'RudriQ DuckDB nodes (LLM side): {len(graph.nodes)}')
for n in graph.nodes:
    print(f'  {n.node_id[:20]:20s} | {n.kind.value:18s} | {n.library}.{n.operation}')

print(f'\nCross-domain edges: {len(graph.edges)}')
for e in graph.edges:
    parent_node = al_tracker.nodes.get(e.parent_id) if al_tracker else None
    parent_label = (
        f'AutoLineage[{parent_node["source"]}, shape={parent_node["shape"]}]'
        if parent_node else f'unknown({e.parent_id[:10]})'
    )
    print(f'  {parent_label}')
    print(f'    -> {e.child_id[:16]} | {e.kind.value} | {e.link_method.value} (conf={e.confidence:.2f})')

if al_tracker:
    print(f'\nUpstream chain (AutoLineage records): {len(al_tracker.records)}')
    for r in al_tracker.records:
        print(f'  {r.library}.{r.operation}  shape: {r.input_shape} -> {r.output_shape}')

## What just happened

RudriQ produced a cross-domain edge connecting the OpenAI call to the upstream DataFrame, with no manual `register_object_identity` and no manual `record_llm_input` in user code. The chain:

1. **AutoLineage** captured pandas operations and assigned each output a lineage ID. Its `register_assign_id_callback` API fired on every assignment, mirroring `id(obj) → lineage_id` into RudriQ's linker registry.
2. **The pandas lid-propagation patch** carried the filtered DataFrame's lineage ID through `df['text']` (a fresh Series) and `.tolist()` (a fresh list) by registering each derivation against the parent's id.
3. **The OpenAI SDK** was monkey-patched at `import rudriq.auto` time. The wrapper extracted the `input=` argument from `embeddings.create` and stashed it against the active OTel span's id.
4. **The SpanProcessor**, on span end, ran the linker against the stashed input. The linker's `link_by_object_identity` strategy found the matching id in step 1+2's registry and emitted a `LINEAGE_LINK` edge with confidence 1.0.

v0.0.5 will mirror AutoLineage's records into RudriQ's DuckDB so the data-side nodes appear in `load_run` directly, eliminating the lookup against `al_tracker.nodes` in cell 4.

v0.1 will ship full Traceloop integration so step 3's span is created automatically by OpenLLMetry instrumentation and the OTel setup cell disappears.